# SportsPulse – IPL Data Analysis with Pandas

**Project:** IPL statistics using `groupby()`, `agg()`, pivot tables, `melt()` and `crosstab()`.

This notebook has been adjusted to work with the two files supplied:
- `deliveries.csv` – ball-by-ball data
- `matches.csv` – match and season information

**Important:** Keep these two CSV files in the same folder as this notebook. Run every cell from top to bottom using **Run → Run All Cells**. The notebook automatically handles the column-name difference (`batter` in your file instead of `batsman`) and joins `match_id` to `id`.


## 1. Import libraries and load the two CSV files

In [ ]:
import pandas as pd
import numpy as np

balls = pd.read_csv("deliveries.csv")
matches = pd.read_csv("matches.csv")

# Clean column headings
balls.columns = balls.columns.str.strip()
matches.columns = matches.columns.str.strip()

# Some IPL files use 'batter'; the assignment calls this column 'batsman'.
if "batsman" not in balls.columns and "batter" in balls.columns:
    balls = balls.rename(columns={"batter": "batsman"})

print("Deliveries shape:", balls.shape)
print("Matches shape:", matches.shape)
print("\nDelivery columns:", balls.columns.tolist())
print("\nMatch columns:", matches.columns.tolist())

display(balls.head())
display(matches.head())

## 2. Prepare the combined dataset

The delivery file identifies a match using `match_id`, while the matches file uses `id`. We join those columns to add the season to every delivery.

In [ ]:
# Check that the columns needed for the join are present
required_ball_columns = ["match_id", "batsman", "bowler", "batting_team",
                         "inning", "over", "batsman_runs", "total_runs",
                         "player_dismissed"]
required_match_columns = ["id", "season"]

missing_balls = [c for c in required_ball_columns if c not in balls.columns]
missing_matches = [c for c in required_match_columns if c not in matches.columns]

if missing_balls or missing_matches:
    raise ValueError(
        f"Missing columns. Deliveries: {missing_balls}; Matches: {missing_matches}"
    )

season_lookup = matches[["id", "season"]].drop_duplicates("id")
data = balls.merge(season_lookup, left_on="match_id", right_on="id", how="left")

# Make run values numeric
for col in ["batsman_runs", "extra_runs", "total_runs"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)

print("Combined rows:", len(data))
print("Seasons found:", sorted(data["season"].dropna().astype(str).unique()))

## Task 1 – Basic groupby

In [ ]:
# 1. Total runs scored by each batsman: top 10
runs_by_batsman = (
    data.groupby("batsman")["batsman_runs"]
        .sum()
        .sort_values(ascending=False)
)

print("Top 10 run-scorers")
display(runs_by_batsman.head(10).reset_index(name="total_runs"))

# 2. Matches played in each season
matches_per_season = (
    matches.groupby("season")["id"]
           .nunique()
           .sort_index()
           .reset_index(name="matches_played")
)

print("Matches played per season")
display(matches_per_season)

# 3. Wins by each team
team_wins = (
    matches["winner"].dropna()
          .value_counts()
          .rename_axis("team")
          .reset_index(name="wins")
)

print("Wins by team")
display(team_wins)

**Explanation:** `groupby()` uses the split–apply–combine process: it splits rows into groups, applies a calculation to each group, and combines the results. For example, grouping by batsman puts each player's deliveries together, `sum()` adds their runs, and Pandas returns one total for each player.

## Task 2 – Multiple aggregations with agg()

In [ ]:
# Calculate each batsman's score in each match first.
# This makes the mean and maximum refer to match scores, not individual balls.
batsman_match_runs = (
    data.groupby(["batsman", "match_id"])["batsman_runs"]
        .sum()
        .reset_index(name="match_runs")
)

batter_summary = (
    batsman_match_runs.groupby("batsman")["match_runs"]
    .agg(
        total_runs="sum",
        average_runs_per_match="mean",
        highest_match_score="max",
        matches_played="count"
    )
    .sort_values("total_runs", ascending=False)
)

print("Top 10 batsmen – multiple statistics")
display(batter_summary.head(10).reset_index())

In [ ]:
# Bowler statistics: count of deliveries, runs conceded and economy rate.
# This follows the assignment formula: runs / (deliveries / 6).
bowler_summary = (
    data.groupby("bowler")
        .agg(
            balls_bowled=("bowler", "count"),
            runs_conceded=("total_runs", "sum")
        )
)

bowler_summary["economy_rate"] = (
    bowler_summary["runs_conceded"] / (bowler_summary["balls_bowled"] / 6)
)

economical_bowlers_300 = (
    bowler_summary[bowler_summary["balls_bowled"] >= 300]
    .sort_values("economy_rate")
)

print("Top 10 economical bowlers (minimum 300 deliveries)")
display(economical_bowlers_300.head(10).reset_index())

**Explanation:** `agg()` allows several statistics to be calculated in one grouped operation. It keeps the code shorter and the results together, instead of repeating a separate `groupby()` for every statistic.

## Task 3 – Pivot tables

In [ ]:
# 1. Total batsman runs by batting team and season
team_season_runs = pd.pivot_table(
    data,
    index="batting_team",
    columns="season",
    values="batsman_runs",
    aggfunc="sum",
    fill_value=0
)

print("Runs scored by each team in each season")
display(team_season_runs)

In [ ]:
# 2. Average total runs per delivery for each over and innings.
# The requested pivot uses total_runs with aggfunc='mean'.
over_innings_average = pd.pivot_table(
    data,
    index="over",
    columns="inning",
    values="total_runs",
    aggfunc="mean"
)

print("Average runs per delivery by over and innings")
display(over_innings_average)

In [ ]:
# 3. Pivot table with row and column totals
team_season_margins = pd.pivot_table(
    data,
    index="batting_team",
    columns="season",
    values="batsman_runs",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="All teams"
)

display(team_season_margins)

# Find the team with the largest total across all seasons
team_totals = team_season_margins.drop(index="All teams")["All teams"]
print("Team with the highest total runs:", team_totals.idxmax())
print("Total runs:", int(team_totals.max()))

**Explanation:** A pivot table displays grouped results in a grid, with one category in rows and another in columns. Use `groupby()` for a simple summary, and use a pivot table when you want to compare categories across rows and columns.

## Task 4 – Reshaping with melt() and crosstab()

In [ ]:
# 1. Wide format to long format
team_season_long = (
    team_season_runs.reset_index()
    .melt(
        id_vars="batting_team",
        var_name="season",
        value_name="total_runs"
    )
    .rename(columns={"batting_team": "team"})
)

display(team_season_long.head(10))

In [ ]:
# 2. Long format back to wide format
team_season_wide_again = (
    team_season_long.pivot(
        index="team",
        columns="season",
        values="total_runs"
    )
    .reindex(index=team_season_runs.index, columns=team_season_runs.columns)
    .fillna(0)
)

print("Original shape:", team_season_runs.shape)
print("Reshaped shape:", team_season_wide_again.shape)
print("Shapes match:", team_season_runs.shape == team_season_wide_again.shape)
display(team_season_wide_again)

In [ ]:
# 3. Toss decision compared with match result
toss_result_table = pd.crosstab(
    matches["toss_decision"],
    matches["result"],
    margins=True
)

display(toss_result_table)

**Crosstab interpretation:** This table counts match results for teams that chose to bat or field after winning the toss. It can show patterns in the dataset, but it does not prove that the toss decision caused a team to win or lose.

**Explanation:** `melt()` is useful when a table has many season columns and you want one season column instead. Long-format data is easier to filter, group, plot, and combine with other data.

## Task 5 – Build a short IPL commentary brief

For this task, the batting average is estimated as total runs divided by the number of times the batter was dismissed. The bowler wicket count follows the assignment instruction and counts non-empty `player_dismissed` entries; it is an approximate count because some dismissals (such as run-outs) are not credited to the bowler.

In [ ]:
# 1. Top five run-scorers with batting average
dismissals_by_batter = (
    data[data["player_dismissed"].notna()]
    .groupby("player_dismissed")
    .size()
)

top5_batters = batter_summary[["total_runs"]].copy()
top5_batters["dismissals"] = (
    top5_batters.index.map(dismissals_by_batter).fillna(0)
)
top5_batters["batting_average"] = (
    top5_batters["total_runs"] /
    top5_batters["dismissals"].replace(0, np.nan)
)
top5_batters = top5_batters.sort_values("total_runs", ascending=False).head(5)

print("Top 5 run-scorers")
display(top5_batters.reset_index())

In [ ]:
# 2. Top five economical bowlers (minimum 500 deliveries) and wickets
wickets_by_bowler = (
    data[data["player_dismissed"].notna()]
    .groupby("bowler")
    .size()
)

bowler_summary["wickets"] = (
    bowler_summary.index.map(wickets_by_bowler).fillna(0).astype(int)
)

top5_bowlers = (
    bowler_summary[bowler_summary["balls_bowled"] >= 500]
    .sort_values("economy_rate")
    .head(5)
)

print("Top 5 economical bowlers (minimum 500 deliveries)")
display(top5_bowlers.reset_index())

In [ ]:
# 3. Season-wise runs, matches and average runs per match
runs_per_season = (
    data.groupby("season")["total_runs"]
        .sum()
        .rename("total_runs")
)

matches_per_season = (
    matches.groupby("season")["id"]
           .nunique()
           .rename("matches_played")
)

season_summary = pd.concat([runs_per_season, matches_per_season], axis=1).fillna(0)
season_summary["average_runs_per_match"] = (
    season_summary["total_runs"] / season_summary["matches_played"]
)

print("Season-wise summary")
display(season_summary.reset_index())

In [ ]:
# 4. Most wins in knockout matches.
# This supplied matches.csv has a match_type column with values such as
# League, Final, Qualifier 1, Qualifier 2, Eliminator and Semi Final.
knockout_matches = matches[
    matches["match_type"].astype(str).str.contains(
        "final|qualifier|eliminator|play-off|playoff",
        case=False,
        na=False
    )
].copy()

knockout_wins = (
    knockout_matches["winner"].dropna()
    .value_counts()
    .rename_axis("team")
    .reset_index(name="knockout_wins")
)

print("Knockout match types found:")
print(sorted(knockout_matches["match_type"].dropna().unique()))

print("Knockout wins by team")
display(knockout_wins)

if not knockout_wins.empty:
    print("Team with most knockout wins:", knockout_wins.iloc[0]["team"])
    print("Number of knockout wins:", int(knockout_wins.iloc[0]["knockout_wins"]))
else:
    print("No knockout matches were identified in the match_type column.")

In [ ]:
# 5. Create a short presenter-style commentary brief from the calculated tables
best_batter = top5_batters.index[0]
best_batter_runs = int(top5_batters.iloc[0]["total_runs"])
best_batter_avg = top5_batters.iloc[0]["batting_average"]

best_bowler = top5_bowlers.index[0]
best_economy = float(top5_bowlers.iloc[0]["economy_rate"])

best_season = season_summary["total_runs"].idxmax()
best_season_runs = int(season_summary.loc[best_season, "total_runs"])
best_season_matches = int(season_summary.loc[best_season, "matches_played"])
best_season_average = float(season_summary.loc[best_season, "average_runs_per_match"])

if not knockout_wins.empty:
    knockout_sentence = (
        f"{knockout_wins.iloc[0]['team']} leads the knockout-win count "
        f"with {int(knockout_wins.iloc[0]['knockout_wins'])} wins in the available data."
    )
else:
    knockout_sentence = "A knockout-wins leader could not be identified from the available match records."

commentary_brief = (
    f"Looking at the IPL records in this dataset, {best_batter} is the leading "
    f"run-scorer with {best_batter_runs:,} runs and an estimated batting average "
    f"of {best_batter_avg:.2f}. "
    f"Among bowlers with at least 500 deliveries, {best_bowler} has the lowest "
    f"economy rate at {best_economy:.2f} runs per over. "
    f"The {best_season} season recorded {best_season_runs:,} runs across "
    f"{best_season_matches} matches, an average of {best_season_average:.1f} "
    f"runs per match. "
    f"{knockout_sentence} "
    f"These numbers give broadcasters a quick view of leading batting, bowling "
    f"and team performances across the seasons covered."
)

print("COMMENTARY BRIEF\n")
print(commentary_brief)

## Final notes

- The supplied files contain seasons beyond 2020 as well. The notebook uses every season present in your CSV files rather than silently removing records.
- The economy rate follows the assignment's requested formula using delivery count divided by six. In official cricket scoring, wides and no-balls are not legal balls, so an official economy calculation can differ.
- The wicket total follows the assignment's requested non-null `player_dismissed` count and may include dismissals not credited to the bowler.
- The over/innings pivot uses the requested mean of `total_runs` for deliveries in each over. This is average runs per delivery, not the total runs in a complete over.
